# RAG System with RAGBench - Using Ingestion Layer and Strategy Classes

This notebook demonstrates the complete RAG pipeline using:
- Ingestion Layer for loading and parsing RAGBench datasets
- Strategy Factory for creating strategies
- RAG Config for configuration
- Complete RAG Pipeline for orchestration

## 1. Setup & Dependencies

In [1]:
!pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk -q

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/adityanarayan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
import json
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq
import numpy as np

# Load environment variables
load_dotenv(override=True)

# Get API keys
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"HuggingFace token loaded: {bool(HF_TOKEN)}")
print(f"Groq API key loaded: {bool(GROQ_API_KEY)}")

HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Configuration

In [9]:
from rag.config.loader import ConfigLoader

# Load the high quality config (with reranking) from its YAML file.
# Providers are built automatically from `config.providers` when the
# pipeline is created, resolving credentials from each provider's `api_key_env`.
CONFIG_PATH = "config/rag_config_high_quality.yaml"
config = ConfigLoader.load(CONFIG_PATH)


print("Configuration:")
print(f"  Name:       {config.name}")
print(f"  Mode:       {config.mode}")
print(f"Providers:")
for provider_name, provider_config in config.providers.items():
    print(f"  {provider_name}: {provider_config.type.value} (api_key_env={provider_config.api_key_env})")
print(f"  Chunking:   {config.chunking.type.value}")
print(f"  Embedding:  {config.embedding.type.value} ({config.embedding.config.model_name})")
print(f"  Retrieval:  {config.retrieval.type.value}")
print(f"  Reranker:   {config.reranker.type.value} ({config.reranker.config.model_name})")
print(f"  Generation: {config.generation.strategy.value} (provider={config.generation.provider}, model={config.generation.config.model})")
print(f"  Evaluation: {config.evaluation.type.value} (provider={config.evaluation.provider})")

Configuration:
  Name:       high-quality
  Mode:       Mode.DEV
Providers:
  groq: groq (api_key_env=GROQ_API_KEY)
  Chunking:   sentence
  Embedding:  sentence_transformer (BAAI/bge-large-en-v1.5)
  Retrieval:  dense_rerank
  Reranker:   cross_encoder (BAAI/bge-reranker-v2-m3)
  Generation: default (provider=groq, model=llama-3.3-70b-versatile)
  Evaluation: trace (provider=groq)


## 4. Load Data Using Data Loader

In [4]:
## 4. Load Data Using Ingestion Layer
from ingestion import (
    RAGBenchCovidQALoader,
    ParserFactory,
    ParserType,
    DataProcessor,
    DatasetLoadingConfig
)

# Create loader for CovidQA dataset
dataset_loading_config = DatasetLoadingConfig(limit=246)  # Full dataset
loader = RAGBenchCovidQALoader(
    split="test",
    config=dataset_loading_config
)

# Load raw data
print("📥 Loading RAGBench CovidQA dataset...")
raw_data = loader.load()
print(f"✅ Loaded {len(raw_data)} samples\n")

# Get dataset info
info = loader.info()
print(f"Dataset Info:")
print(f"  Source: {info['source']}")
print(f"  Dataset: {info['dataset_name']}")
print(f"  Split: {info['split']}")
print(f"  Samples: {info['num_samples']}")
print(f"  Keys: {len(info['keys'])} fields\n")

# Inspect first sample
first_sample = loader.load_sample(0)
print(f"First Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")
print(f"  Has ground truth scores: {bool(first_sample.get('relevance_score'))}")

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading RAGBench CovidQA dataset...
Loading RAGBench dataset: covidqa (test)...
Loaded 246 samples
✅ Loaded 246 samples

Dataset Info:
  Source: galileo-ai/ragbench
  Dataset: covidqa
  Split: test
  Samples: 246
  Keys: 26 fields

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4
  Has ground truth scores: True


## 5. Parse Documents

In [5]:
## 5. Parse Documents Using Ingestion Layer
# Create parser strategy
print("📄 Creating parser strategy...")
parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
print(f"✅ Parser created: {parser.__class__.__name__}\n")

# Create data processor with parser strategy
processor = DataProcessor(parser_strategy=parser)

# Process dataset into canonical Document objects
print("📄 Processing documents...")
documents = processor.process_dataset(raw_data)
print(f"✅ Processed {len(documents)} documents\n")

# Print sample
sample_doc = documents[0]
print(f"Sample RAG Document:")
print(f"  Title: {sample_doc.title[:60]}...")
print(f"  Content: {sample_doc.content[:100]}...")
print(f"  Metadata: {sample_doc.metadata}")

📄 Creating parser strategy...
✅ Parser created: TitlePassageParser

📄 Processing documents...
✅ Processed 984 documents

Sample RAG Document:
  Title: Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  Content: successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV m...
  Metadata: {'doc_id': 'sample_0_doc_0', 'sample_index': 0, 'source': 'ragbench', 'parser_type': 'title_passage'}


## 6. Models Are Config-Driven

Both the embedding model and the reranking model are created inside the pipeline from the RAG config (`config.embedding` / `config.reranker`). No manual model loading is required, and no API clients are passed in: providers are built automatically from `config.providers`, resolving credentials from each provider's `api_key_env`.

In [6]:
# Both the embedding model and the reranking model are created inside the
# pipeline from the RAG config (config.embedding / config.reranker).
# No manual model loading or API clients are needed here -- providers are
# built automatically from config.providers when the pipeline is created.
print("Embedding + reranking models will be built by the pipeline from config.")

Embedding + reranking models will be built by the pipeline from config.


## 7. Initialize RAG Pipeline

In [7]:
from rag.pipeline.rag_pipeline import RAGPipeline

# The pipeline is a pure orchestrator. It builds every strategy from the config
# object and registers providers automatically from config.providers -- no
# strategy-specific parameters or API clients are passed in here.
print("Initializing RAG Pipeline...")
pipeline = RAGPipeline(config)
print("Pipeline initialized\n")

# Verify strategies were created
print("Strategies created:")
print(f"  Chunker:   {pipeline.chunker.__class__.__name__}")
print(f"  Embedder:  {pipeline.embedder.__class__.__name__}")
print(f"  Reranker:  {pipeline.reranker.__class__.__name__}")
print(f"  Retriever: {pipeline.retriever.__class__.__name__}")
print(f"  Generator: {pipeline.generator.__class__.__name__}")
print(f"  Evaluator: {pipeline.evaluator.__class__.__name__}")

2026-06-21 20:52:45,382 DEBUG rag.pipeline.rag_pipeline: Initializing RAG pipeline in dev mode


Initializing RAG Pipeline...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7793.16it/s]


Pipeline initialized

Strategies created:
  Chunker:   SentenceChunkingStrategy
  Embedder:  SentenceTransformerEmbeddingStrategy
  Reranker:  CrossEncoderRerankerStrategy
  Retriever: DenseRerankRetrievalStrategy
  Generator: DefaultGenerationStrategy
  Evaluator: TRACeEvaluationStrategy


## 8. Build Vector Index

In [10]:
# Build index - this will chunk, embed, and store
print("🏗️  Building vector index...")
pipeline.build_index(documents)
print(f"✅ Index built and ready for querying")

2026-06-21 20:55:14,106 INFO rag.pipeline.rag_pipeline: Processing 984 documents...
2026-06-21 20:55:14,116 INFO rag.pipeline.rag_pipeline: [Chunk Cache] HIT key=a7c984e496237e69870e35aebde38527420d7c2e4e6cb3e3beb827e63656ab64
2026-06-21 20:55:14,116 DEBUG rag.pipeline.rag_pipeline: Chunk cache key=a7c984e496237e69870e35aebde38527420d7c2e4e6cb3e3beb827e63656ab64 (984 chunks)
2026-06-21 20:55:14,127 INFO rag.pipeline.rag_pipeline: [Embedding Cache] HIT key=53fa1b16676490718b5d1ab49d647571ed4624fa8eb217ab4b4184e4d027910c
2026-06-21 20:55:14,128 DEBUG rag.pipeline.rag_pipeline: Embedding cache key=53fa1b16676490718b5d1ab49d647571ed4624fa8eb217ab4b4184e4d027910c (984 vectors)
2026-06-21 20:55:14,135 INFO rag.pipeline.rag_pipeline: [Index Cache] HIT key=b3c3ecceb4ad62ed8cd0d0f0417c9710ca99ba2f8283a466b4d103076ac88bd8
2026-06-21 20:55:14,136 DEBUG rag.pipeline.rag_pipeline: Index cache key=b3c3ecceb4ad62ed8cd0d0f0417c9710ca99ba2f8283a466b4d103076ac88bd8
2026-06-21 20:55:14,136 INFO rag.pipel

🏗️  Building vector index...
✅ Index built and ready for querying


## 9. Test Single Query

In [11]:
# Test with first sample
test_sample = raw_data[0]
test_query = test_sample['question']

print(f"❓ Test Query:")
print(f"   {test_query}\n")

# Run query through pipeline
result = pipeline.query(test_query)

# Display results
print(f"\n📊 Query Results:")
print(f"\nRetrieved Documents: {len(result['retrieved_docs'])}")
for i, doc in enumerate(result['retrieved_docs'][:3], 1):
    print(f"  {i}. {doc['metadata']['title'][:60]}...")

print(f"\n💬 Generated Response:")
print(result['response'][:300] + "...\n")

print(f"📈 TRACe Scores:")
scores = result['scores']
print(f"  Relevance:    {scores['relevance_score']:.4f}")
print(f"  Utilization:  {scores['utilization_score']:.4f}")
print(f"  Completeness: {scores['completeness_score']:.4f}")
print(f"  Adherence:    {scores['adherence_score']}")

2026-06-21 20:55:19,987 INFO rag.pipeline.rag_pipeline: Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
2026-06-21 20:55:19,987 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...


❓ Test Query:
   Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?



2026-06-21 20:55:22,522 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-21 20:55:22,523 DEBUG rag.pipeline.rag_pipeline: Generating response...
2026-06-21 20:55:23,370 DEBUG rag.pipeline.rag_pipeline: Response generated (686 chars)
2026-06-21 20:55:23,370 DEBUG rag.pipeline.rag_pipeline: Evaluating response...
2026-06-21 20:55:25,490 DEBUG rag.pipeline.rag_pipeline: Evaluation complete



📊 Query Results:

Retrieved Documents: 5
  1. Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  2. Depletion of Alveolar Macrophages Does Not Prevent Hantaviru...
  3. Confounding roles for type I interferons during bacterial an...

💬 Generated Response:
Based on the context provided, it appears that viruses such as HCV (Hepatitis C Virus) may persist due to the loss of innate immune function, suggesting that they can cause prolonged inflammation. 

On the other hand, viruses like TBFV (Tick-borne flavivirus) trigger a strong antiviral response, inc...

📈 TRACe Scores:
  Relevance:    0.6522
  Utilization:  0.2609
  Completeness: 0.4000
  Adherence:    False


## 10. Generate Detailed Report with ReportGenerator

The `ReportGenerator` is a thin orchestrator over pluggable `ReportStrategy`
implementations. We use the bundled `DetailedQueryReportStrategy`, which builds:

- a **per-query table**: query, retrieved documents as an array **with all
  retrieval scores**, the generated answer, and every TRACe score next to its
  dataset ground truth plus the per-query deviation; and
- an **aggregate table**: per TRACe metric, the mean score, mean ground truth,
  standard deviation of each, and mean absolute error vs. ground truth.

It reuses the `pipeline`, `config`, and `raw_data` already built above. New
report formats can be added by implementing `ReportStrategy` and registering it
with the generator — no change to `ReportGenerator` itself.

In [12]:
from rag.reporting import (
    DetailedQueryReportStrategy,
    ReportGenerator,
)

from rag.models.pipeline_run_result import PipelineRunResult
from rag.models.query_result import QueryResult

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# Number of queries to include in the report.
NUM_REPORT_QUERIES = 5

records = []
for sample in raw_data[:NUM_REPORT_QUERIES]:
    query = sample["question"]
    result = pipeline.query(query)  # retrieved_docs carry all retrieval scores
    records.append(
        QueryResult(
            query=query,
            retrieved_docs=result["retrieved_docs"],
            answer=result["response"],
            metadata={
                "predicted_scores": result["scores"],
                "ground_truth_scores": {
                    "relevance_score": sample.get("relevance_score", 0.0),
                    "utilization_score": sample.get("utilization_score", 0.0),
                    "completeness_score": sample.get("completeness_score", 0.0),
                    "adherence_score": sample.get("adherence_score", False),
                },
            },
            
        )
    )

run = PipelineRunResult(
    records=records,
    config=config
)

generator = ReportGenerator(DetailedQueryReportStrategy())
print("Available report strategies:", generator.strategies)

report = generator.generate(run)
report.display()

2026-06-21 20:55:29,888 INFO rag.pipeline.rag_pipeline: Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
2026-06-21 20:55:29,888 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...
2026-06-21 20:55:31,440 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-21 20:55:31,440 DEBUG rag.pipeline.rag_pipeline: Generating response...
2026-06-21 20:55:32,275 DEBUG rag.pipeline.rag_pipeline: Response generated (1066 chars)
2026-06-21 20:55:32,276 DEBUG rag.pipeline.rag_pipeline: Evaluating response...
2026-06-21 20:55:33,887 DEBUG rag.pipeline.rag_pipeline: Evaluation complete
2026-06-21 20:55:33,888 INFO rag.pipeline.rag_pipeline: Query: When was the first case of COVID-19 identified?
2026-06-21 20:55:33,889 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...
2026-06-21 20:55:35,337 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-21 20:55:35,338 DEBUG rag.pipeline.rag_pipeline: Generating response...
2

AllKeysExhaustedException: All Groq API keys are currently rate limited.